<a href="https://colab.research.google.com/github/Reito-alt/AI-Tuning/blob/main/tes_dataset_qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install Library**

In [1]:
!pip install -q transformers==4.46.3 peft==0.13.2 trl==0.12.1 bitsandbytes accelerate datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


**Upload Dataset**

In [1]:
from google.colab import files
uploaded = files.upload()
DATASET_PATH = list(uploaded.keys())[0]
print("Dataset:", DATASET_PATH)

Saving dataset_for_qwen.jsonl to dataset_for_qwen.jsonl
Dataset: dataset_for_qwen.jsonl


**Skrip Training QLoRA**

In [2]:
import json, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "qwen-asisten-belajar-lora"
MAX_SEQ_LEN = 512

# 1. Load dataset dan bagi 90:10 (train/test)
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(dataset)

# 2. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 3. Load model dalam 4-bit (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

# 4. Konfigurasi LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# 5. Argumen training
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

# 6. Format percakapan memakai chat template Qwen
def formatting_func(example):
    return tokenizer.apply_chat_template(example["messages"], tokenize=False)

# 7. Trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=lora_config,
    processing_class=tokenizer,
    formatting_func=formatting_func,
    max_seq_length=MAX_SEQ_LEN,
)

# 8. Mulai training
trainer.train()

# 9. Simpan adapter LoRA
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Adapter tersimpan di:", OUTPUT_DIR)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 54
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 6
    })
})


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (7 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
0,1.815400,1.161418
1,1.246100,0.999816
2,1.149300,0.995616


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Adapter tersimpan di: qwen-asisten-belajar-lora


**Uji Coba Model**

In [3]:
from transformers import pipeline

pipe = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer)
messages = [
    {"role": "system", "content": "Kamu adalah asisten belajar yang sabar, jelas, dan menjawab dalam bahasa Indonesia untuk siswa SMA dan mahasiswa."},
    {"role": "user", "content": "Jelaskan apa itu fotosintesis secara singkat."},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
out = pipe(prompt, max_new_tokens=200, do_sample=False, return_full_text=False)
print(out[0]["generated_text"])

The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'GitForCausalLM', 'GlmForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'GraniteForCausalLM', 'GraniteMoeForCausalLM', 'JambaForCausalLM', 'JetMoeForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'Mamba2ForCausalLM', 'MarianForCausalLM', 'MBartForCausa

Fotosintesis adalah proses kimia yang dilakukan oleh tumbuhan, di mana mereka mengubah energi matahari menjadi makanan (glukosa) yang bisa digunakan selama bertahun-tahun. Proses ini terjadi pada bagian bawah daun atau akar tanaman, dimana ada organisme khusus disebut plastid. Dalam proses ini, air, oksigen, dan CO2 dari udara dicampur dengan zat gula alami (glukosa) yang diperoleh dari cahaya matahari. Glukosa kemudian membantu tumbuhan mempertahankan diri, menumpukan energi, dan mendapatkan nutrisi lainnya. Proses ini sangat penting karena memberikan sumber energi bagi kehidupan hidup di permukaan bumi.


**Save File Training**

In [4]:
!zip -r qwen-asisten-belajar-lora.zip qwen-asisten-belajar-lora
from google.colab import files
files.download("qwen-asisten-belajar-lora.zip")

  adding: qwen-asisten-belajar-lora/ (stored 0%)
  adding: qwen-asisten-belajar-lora/special_tokens_map.json (deflated 63%)
  adding: qwen-asisten-belajar-lora/tokenizer.json (deflated 81%)
  adding: qwen-asisten-belajar-lora/checkpoint-6/ (stored 0%)
  adding: qwen-asisten-belajar-lora/checkpoint-6/special_tokens_map.json (deflated 63%)
  adding: qwen-asisten-belajar-lora/checkpoint-6/tokenizer.json (deflated 81%)
  adding: qwen-asisten-belajar-lora/checkpoint-6/vocab.json (deflated 61%)
  adding: qwen-asisten-belajar-lora/checkpoint-6/adapter_config.json (deflated 53%)
  adding: qwen-asisten-belajar-lora/checkpoint-6/added_tokens.json (deflated 67%)
  adding: qwen-asisten-belajar-lora/checkpoint-6/merges.txt (deflated 57%)
  adding: qwen-asisten-belajar-lora/checkpoint-6/rng_state.pth (deflated 26%)
  adding: qwen-asisten-belajar-lora/checkpoint-6/scheduler.pt (deflated 61%)
  adding: qwen-asisten-belajar-lora/checkpoint-6/tokenizer_config.json (deflated 83%)
  adding: qwen-asisten-b

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>